# Situational Awareness Portfolio

## Project Statement

This project starts with the fictional Situational Awareness portfolio introduced in `Leveraged Awareness`: long memory, networking, and data-center infrastructure; short application software. Equal sleeves keep the view transparent, but they ignore differences in estimated risk and return. Preserve the mandate, build one stabilized optimizer, and decide whether it improves on equal sleeves after 2024.

## Resources

### Course Materials

- `Risk and Return Metrics`: annualization, Sharpe ratio, and drawdown.
- `Optimizing Risk and Return`: estimated tangency portfolios and constraints.
- `MV of S&P500`: unstable inputs and portfolio weights.
- `Leveraged Awareness`: the long-short mandate, the July 2026 reversal, and return attribution.

### Data

| File | Sheets | Frequency | Sample |
|---|---|---|---|
| `project_situational_awareness_20260731.xlsx` | `descriptions`, `total return indexes`, `total returns`, `baseline weights`, `data source`, `checks` | Daily | 2020-12-31 to 2026-07-31 |

Bloomberg gross-dividend total-return indexes supply the return history. The `SNDK` strategy slot uses `WDC` before Sandisk begins regular-way trading and `SNDK` thereafter. The panel provides four years before the portfolio test and is separate from the shorter case dataset.

## Data Preview

In [5]:
from IPython.display import display
from pathlib import Path
import pandas as pd

DATA = Path('project_situational_awareness_20260731.xlsx')
descriptions = pd.read_excel(DATA, sheet_name='descriptions').set_index('ticker')
returns = pd.read_excel(DATA, sheet_name='total returns', index_col='Date', parse_dates=True)
baseline = pd.read_excel(DATA, sheet_name='baseline weights').set_index('ticker')

display(descriptions)
baseline_preview = baseline[['side', 'equal-sleeve weight']].copy()
baseline_preview['position on $100m NAV'] = 100_000_000 * baseline_preview['equal-sleeve weight']
display(baseline_preview)
display(returns.iloc[:3, :8])
display(returns.iloc[-3:, :8])

,name,theme,side
ticker,,,
SNDK,Sandisk,memory and storage,long
MU,Micron Technology,memory,long
VRT,Vertiv,data-center power and cooling,long
LITE,Lumentum,optical networking,long
COHR,Coherent,optical networking,long
BE,Bloom Energy,data-center power,long
ADBE,Adobe,application software,short
CRM,Salesforce,application software,short
ORCL,Oracle,enterprise software,short


,side,equal-sleeve weight,position on $100m NAV
ticker,,,
SNDK,long,0.333333,3.333333e+07
MU,long,0.333333,3.333333e+07
VRT,long,0.333333,3.333333e+07
LITE,long,0.333333,3.333333e+07
COHR,long,0.333333,3.333333e+07
BE,long,0.333333,3.333333e+07
ADBE,short,-0.333333,-3.333333e+07
CRM,short,-0.333333,-3.333333e+07
ORCL,short,-0.333333,-3.333333e+07


,SNDK,MU,VRT,LITE,COHR,BE,ADBE,CRM
Date,,,,,,,,
2021-01-04,-0.057050,-0.015031,-0.008034,0.020253,0.000658,-0.047802,-0.029553,-0.009976
2021-01-05,0.015891,0.043349,-0.019978,0.048180,0.040521,0.039209,0.000721,0.005492
2021-01-06,0.006596,-0.001941,0.005510,0.012823,0.003793,0.072638,-0.039902,-0.024242


,SNDK,MU,VRT,LITE,COHR,BE,ADBE,CRM
Date,,,,,,,,
2026-07-29,-0.073178,-0.099360,-0.172578,-0.076051,-0.087453,-0.018521,0.057188,0.037907
2026-07-30,0.259940,0.183569,0.019996,0.150892,0.121639,0.264855,-0.058953,-0.040716
2026-07-31,-0.050884,-0.059023,0.061846,0.029860,0.055529,-0.006325,0.010125,0.018317


## Portfolio Timing

Use the most recent 756 daily returns at each estimate date. Estimate the first weights at the December 31, 2024 close. Re-estimate at each later month-end through June 30, 2026 and hold each weight vector over the following month's daily returns. Estimate once more on July 31, 2026 for the final allocation decision.

A weight set at the close of date $t$ first earns the return on the next trading day. Reset the equal-sleeve comparison on the same month-end dates. Ignore financing costs, transaction costs, taxes, and market impact.

# 1 The Mandate

## 1.1

Verify the equal-sleeve portfolio's long, short, gross, and net exposures. Report the position in each stock for $100 million of NAV.

In [15]:
NAV = 100_000_000

weights = baseline['equal-sleeve weight']

# Exposure calculations
long_exposure = weights[weights > 0].sum()
short_exposure = weights[weights < 0].sum()
gross_exposure = weights.abs().sum()
net_exposure = weights.sum()

# Dollar positions
positions = baseline[['name', 'side', 'equal-sleeve weight']].copy()
positions['position on $100m NAV'] = NAV * positions['equal-sleeve weight']

In [16]:
exposure_summary = pd.DataFrame({
    'Exposure': [
        long_exposure,
        short_exposure,
        gross_exposure,
        net_exposure
    ],
    'Dollar Exposure': [
        long_exposure * NAV,
        short_exposure * NAV,
        gross_exposure * NAV,
        net_exposure * NAV
    ]
}, index=['Long', 'Short', 'Gross', 'Net'])

In [17]:
display(positions)


,name,side,equal-sleeve weight,position on $100m NAV
ticker,,,,
SNDK,Sandisk,long,0.333333,3.333333e+07
MU,Micron Technology,long,0.333333,3.333333e+07
VRT,Vertiv,long,0.333333,3.333333e+07
LITE,Lumentum,long,0.333333,3.333333e+07
COHR,Coherent,long,0.333333,3.333333e+07
BE,Bloom Energy,long,0.333333,3.333333e+07
ADBE,Adobe,short,-0.333333,-3.333333e+07
CRM,Salesforce,short,-0.333333,-3.333333e+07
ORCL,Oracle,short,-0.333333,-3.333333e+07


In [20]:
display(
    exposure_summary.style.format({
        'Exposure': '{:.0%}',
        'Dollar Exposure': '${:,.0f}'
    })
)

,Exposure,Dollar Exposure
Long,200%,"$200,000,000"
Short,-200%,"$-200,000,000"
Gross,400%,"$400,000,000"
Net,0%,$0


The baseline portfolio contains **six long positions** and **six short positions**, with each security assigned an absolute weight of approximately

$$
\frac{1}{3} = 33.33\%
$$

of portfolio NAV.

Therefore, total long exposure is

$$
6\left(\frac{1}{3}\right) = 2 = 200\%
$$

while total short exposure is

$$
6\left(-\frac{1}{3}\right) = -2 = -200\%.
$$

Gross exposure measures the total absolute exposure of the portfolio:

$$
\text{Gross Exposure}
=
\sum_i |w_i|
=
200\% + 200\%
=
400\%.
$$

Net exposure preserves the signs of the positions:

$$
\text{Net Exposure}
=
\sum_i w_i
=
200\% - 200\%
=
0\%.
$$

For a portfolio with **\$100 million NAV**, each position represents approximately

$$
\$100\text{M}\times\frac{1}{3}
=
\$33.33\text{M}.
$$

Thus, the portfolio holds approximately **\$200 million in long positions** and **\$200 million in short positions**.

Although the portfolio has **0% net exposure**, it is not risk-free. Its **400% gross exposure** means that it has substantial exposure on both sides of the market, so losses can occur if the long positions fall, the short positions rise, or both occur simultaneously.


## 1.2

Using only returns available through December 31, 2024, report annualized mean, volatility, Sharpe ratio, and the correlation matrix for the 12 stocks. Identify one feature of the estimates that could make an unconstrained optimizer unreliable.

In [23]:
# Historical estimates through December 31, 2024

CUTOFF_ROW = pd.Timestamp('2024-12-31')
TRADING_DAYS = 252

tickers = baseline.index
historical_returns = returns.loc[:CUTOFF_ROW, tickers].copy()



In [24]:

# Annualized statistics
annualized_mean = historical_returns.mean() * TRADING_DAYS

annualized_volatility = (
    historical_returns.std() * (TRADING_DAYS ** 0.5)
)

# Zero risk-free rate
sharpe_ratio = annualized_mean / annualized_volatility


In [25]:

historical_stats = pd.DataFrame({
    'Annualized Mean': annualized_mean,
    'Annualized Volatility': annualized_volatility,
    'Sharpe Ratio': sharpe_ratio
})

display(
    historical_stats.style.format({
        'Annualized Mean': '{:.2%}',
        'Annualized Volatility': '{:.2%}',
        'Sharpe Ratio': '{:.2f}'
    })
)


,Annualized Mean,Annualized Volatility,Sharpe Ratio
ticker,,,
SNDK,10.63%,42.00%,0.25
MU,13.10%,44.18%,0.30
VRT,61.66%,56.45%,1.09
LITE,7.55%,45.69%,0.17
COHR,19.93%,53.42%,0.37
BE,23.58%,79.60%,0.30
ADBE,3.71%,36.24%,0.10
CRM,16.91%,36.08%,0.47
ORCL,29.87%,30.56%,0.98


In [29]:
# Correlation Mat
correlation_matrix = historical_returns.corr()

display(
    correlation_matrix.style
        .format('{:.2f}')
        .background_gradient(
            cmap='RdBu',
            vmin=-1,
            vmax=1,
            axis=None
        )
)

ticker,SNDK,MU,VRT,LITE,COHR,BE,ADBE,CRM,ORCL,NOW,WDAY,MDB
ticker,,,,,,,,,,,,
SNDK,1.00,0.70,0.40,0.43,0.46,0.29,0.35,0.35,0.29,0.31,0.30,0.31
MU,0.70,1.00,0.42,0.44,0.50,0.32,0.40,0.37,0.29,0.38,0.37,0.36
VRT,0.40,0.42,1.00,0.37,0.44,0.29,0.37,0.38,0.33,0.41,0.31,0.38
LITE,0.43,0.44,0.37,1.00,0.61,0.29,0.34,0.38,0.29,0.35,0.34,0.31
COHR,0.46,0.50,0.44,0.61,1.00,0.31,0.37,0.36,0.30,0.39,0.36,0.37
BE,0.29,0.32,0.29,0.29,0.31,1.00,0.26,0.28,0.17,0.31,0.31,0.34
ADBE,0.35,0.40,0.37,0.34,0.37,0.26,1.00,0.61,0.40,0.66,0.57,0.51
CRM,0.35,0.37,0.38,0.38,0.36,0.28,0.61,1.00,0.36,0.69,0.61,0.50
ORCL,0.29,0.29,0.33,0.29,0.30,0.17,0.40,0.36,1.00,0.38,0.31,0.26


Using returns available through **December 31, 2024**, we estimate each stock's annualized mean return, annualized volatility, Sharpe ratio, and pairwise return correlations.

The historical estimates vary substantially across securities. **VRT** had the highest estimated annualized mean return at **61.66%** and also the highest Sharpe ratio at **1.09**, despite relatively high annualized volatility of **56.45%**. **ORCL** also had strong historical risk-adjusted performance, with an annualized return of **29.87%**, volatility of **30.56%**, and Sharpe ratio of **0.98**.

At the other extreme, some securities produced considerably weaker historical risk-adjusted returns. **ADBE** had an estimated annualized mean return of only **3.71%** and a Sharpe ratio of **0.10**, while **LITE** had a Sharpe ratio of **0.17**. **BE** exhibited the highest volatility in the portfolio at **79.60%**, while **MDB** was also highly volatile at **68.74%**.

The correlation matrix shows that most securities are positively correlated, although the strength of these relationships differs across the portfolio. Some of the strongest relationships occur between economically similar stocks. For example:

- **SNDK and MU:** correlation of **0.70**
- **LITE and COHR:** correlation of **0.61**
- **ADBE and NOW:** correlation of **0.66**
- **CRM and NOW:** correlation of **0.69**
- **NOW and WDAY:** correlation of **0.66**

These are important because our portfolio risk depends not only on the volatility of each individual security, but also on how the securities move relative to one another.

#### Potential Problem for an Unconstrained Optimizer

One feature that could make an unconstrained optimizer unreliable is the **large dispersion in estimated historical mean returns**. For example, VRT's estimated annualized mean return of **61.66%** is dramatically higher than estimates such as **7.55% for LITE** or **3.71% for ADBE**.

These sample means are estimates based on historical data rather than known future expected returns. If part of VRT's unusually high historical return is due to sampling variation or a temporary period of exceptional performance, an unconstrained Sharpe-ratio optimizer may place an excessively large weight on VRT because it treats the estimated mean as if it were the true expected return.


# 2 The Optimization Rule

Choose one optimization rule.

- **Position cap.** Use the sample mean and covariance matrix. Constrain every absolute position to no more than 50% of NAV.
- **Covariance shrinkage.** Use the sample mean. Keep every sample variance and divide every off-diagonal covariance by four. Do not add a position cap.

For either rule, maximize the estimated Sharpe ratio with a zero risk-free rate. Long weights must be nonnegative and sum to +2. Short weights must be nonpositive and sum to -2. The resulting portfolio therefore keeps 400% gross exposure and zero net exposure. Use the equal-sleeve weights as the optimizer's starting point.

## 2.1

State your chosen rule, the objective, and every constraint. Explain what the rule is meant to stabilize. Verify numerically that the optimizer succeeds and that its output satisfies the mandate.

## 2.2

Before calculating any return after 2024, write two numeric pass-fail criteria for replacing equal sleeves: one concerning the optimized weights or turnover, and one concerning realized return or risk. Explain why each threshold would matter to this portfolio. Do not revise the criteria after seeing the test results.

# 3 The First Weights

## 3.1

Use the 756 returns ending December 31, 2024 to estimate the optimized weights. Compare them with equal sleeves. Report long, short, gross, and net exposure; the largest absolute position; and the estimated annualized mean, volatility, and Sharpe ratio.

## 3.2

Identify the two largest changes from equal sleeves. Trace each change to estimated mean, volatility, or covariance rather than describing the company alone.

# 4 Rolling Weights

## 4.1

Re-estimate the optimized portfolio at each month-end in the stated test period. Plot all 12 weights. Verify that every weight vector satisfies the mandate and, if applicable, the position cap.

## 4.2

Report each stock's minimum and maximum weight and the portfolio's annualized one-way turnover. Identify the two largest month-to-month changes in the weight vector and trace them to changes in the estimates.

# 5 After 2024

## 5.1

From January 2, 2025 through July 31, 2026, compare the monthly rebalanced equal-sleeve and optimized portfolios. Report cumulative return, annualized mean, annualized volatility, Sharpe ratio, maximum drawdown, and worst day. Plot the NAV of $100 million invested in each.

## 5.2

Identify one period in which the two paths separate. Attribute the difference to weights and stock returns. State whether the optimizer benefited from information available before the period or from an outcome it could not have anticipated.

# 6 The Reversal and the Allocation

## 6.1

For the July 24 through July 29, 2026 reversal, report the return and dollar P&L of each portfolio. Attribute each result to the long sleeve, short sleeve, and three largest stock contributions. Explain which earlier weight decisions mattered most.

## 6.2

Apply the two criteria declared in Section 2. Choose equal sleeves or your optimized allocation at the July 31, 2026 close. Report the weights you would use. Name one observable result over the next year that would reverse the choice and state how you would measure it.

Follow [Project Guidelines](../Project%20Guidelines.md).